# Fine-tuning MMS ASR para K'iche' (LoRA)

Este notebook entrena el modelo de reconocimiento de voz de Meta (MMS) para mejorar la detección de audio en K'iche'.

**Requisitos:**
- Google Colab con GPU (Runtime > Change runtime type > T4 GPU)
- Carpeta `training_data/` con audios `.wav` y `metadata.csv`

**Formato de metadata.csv:**
```
file_name,transcription,asr_raw
audio/sample_0001.wav,k'ax waqan,Q'ax wakan
```

## 1. Configuración inicial

In [ ]:
# Verificar que tenemos GPU
!nvidia-smi
import torch
print(f"GPU disponible: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

In [ ]:
# Instalar dependencias
!pip install -q transformers datasets librosa soundfile peft==0.11.1 accelerate evaluate jiwer tensorboard torchao>=0.16.0

## 2. Subir datos de entrenamiento

Sube tu carpeta `training_data.zip` que contiene:
```
training_data/
  metadata.csv
  audio/
    sample_0001.wav
    sample_0002.wav
    ...
```

**Opción A**: Sube el ZIP manualmente (botón de archivos a la izquierda)

**Opción B**: Desde Google Drive

In [ ]:
# === OPCIÓN A: Subir ZIP directamente ===
# Descomenta las siguientes líneas si subes el ZIP manualmente

# from google.colab import files
# uploaded = files.upload()  # Selecciona training_data.zip
# !unzip -o training_data.zip -d /content/

# === OPCIÓN B: Desde Google Drive ===
# Descomenta si lo subes a Drive primero

from google.colab import drive
drive.mount('/content/drive')

# Cambia esta ruta a donde subiste el ZIP en Drive
DRIVE_ZIP_PATH = '/content/drive/MyDrive/traductor_kiche/training_data.zip'

import os
if os.path.exists(DRIVE_ZIP_PATH):
    !unzip -o "{DRIVE_ZIP_PATH}" -d /content/
    print("Datos extraídos correctamente.")
else:
    print(f"No se encontró {DRIVE_ZIP_PATH}")
    print("Sube el ZIP a Google Drive o usa la Opción A.")

In [ ]:
# Verificar datos
import pandas as pd
import os

DATA_DIR = '/content/training_data'
CSV_PATH = os.path.join(DATA_DIR, 'metadata.csv')

df = pd.read_csv(CSV_PATH, keep_default_na=False)
print(f"Total muestras en CSV: {len(df)}")

# Verificar que existen los archivos de audio
df['exists'] = df['file_name'].apply(lambda x: os.path.exists(os.path.join(DATA_DIR, x)))
valid = df[df['exists']]
print(f"Muestras con audio válido: {len(valid)}")

if len(valid) < 20:
    print("\n⚠️  Necesitas al menos 20 muestras con audio para entrenar.")
    print("Usa la interfaz /train de tu app para grabar más muestras.")
else:
    print(f"\n✅ Datos suficientes para comenzar el entrenamiento.")

print("\nEjemplos:")
print(valid[['file_name', 'transcription']].head(10))

## 3. Preparar Dataset

In [ ]:
import librosa
import numpy as np
from datasets import Dataset, Audio

DATA_DIR = '/content/training_data'

# Cargar CSV y filtrar solo muestras con audio existente
df = pd.read_csv(os.path.join(DATA_DIR, 'metadata.csv'), keep_default_na=False)
df['full_path'] = df['file_name'].apply(lambda x: os.path.join(DATA_DIR, x))
df = df[df['full_path'].apply(os.path.exists)].reset_index(drop=True)

print(f"Muestras válidas: {len(df)}")

# Crear dataset de HuggingFace
dataset = Dataset.from_dict({
    'audio': df['full_path'].tolist(),
    'transcription': df['transcription'].tolist()
}).cast_column('audio', Audio(sampling_rate=16000))

# Split 90/10
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset = split['test']

print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

## 4. Cargar modelo MMS y configurar para K'iche'

In [ ]:
from transformers import Wav2Vec2ForCTC, AutoProcessor

MODEL_ID = "facebook/mms-1b-all"
TARGET_LANG = "quc-dialect_central"

print("Cargando modelo MMS (esto toma ~2-3 minutos)...")
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_ID)

# Configurar para K'iche'
processor.tokenizer.set_target_lang(TARGET_LANG)
model.load_adapter(TARGET_LANG)

# Congelar el modelo base - solo entrenaremos el adaptador
model.freeze_base_model()

print("Modelo cargado y configurado para K'iche'.")
print(f"Parámetros totales: {sum(p.numel() for p in model.parameters()):,}")
print(f"Parámetros entrenables: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 5. Configurar LoRA para entrenamiento eficiente

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# Primero descongelar para que LoRA pueda aplicarse
for param in model.parameters():
    param.requires_grad = False

# Configuración LoRA - conservadora para pocos datos
lora_config = LoraConfig(
    r=16,                          # Rango (bajo = menos parámetros)
    lora_alpha=32,                 # Factor de escala
    lora_dropout=0.1,              # Regularización
    target_modules=["k_proj", "v_proj", "q_proj", "out_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Mover a GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Modelo en: {device}")

## 6. Preprocesar audio para el modelo

In [ ]:
import torch
from dataclasses import dataclass
from typing import Dict, List, Union

def prepare_dataset(batch):
    audio = batch['audio']
    batch['input_values'] = processor(
        audio['array'],
        sampling_rate=audio['sampling_rate'],
        return_tensors='pt'
    ).input_values[0]

    batch['labels'] = processor.tokenizer(
        batch['transcription'],
        return_tensors='pt'
    ).input_ids[0]

    return batch

print("Preprocesando train...")
train_dataset = train_dataset.map(prepare_dataset, remove_columns=['audio', 'transcription'])
print("Preprocesando eval...")
eval_dataset = eval_dataset.map(prepare_dataset, remove_columns=['audio', 'transcription'])

print(f"\n✅ Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: AutoProcessor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_values = [{'input_values': f['input_values']} for f in features]
        label_features = [{'input_ids': f['labels']} for f in features]

        batch = self.processor.pad(input_values, padding=self.padding, return_tensors='pt')

        labels_batch = self.processor.tokenizer.pad(label_features, padding=self.padding, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)

        batch['labels'] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)

## 7. Configurar métrica y entrenar

In [ ]:
import evaluate
import numpy as np

wer_metric = evaluate.load('wer')

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {'wer': wer}

In [ ]:
from transformers import TrainingArguments, Trainer

# Ajustar epochs según cantidad de datos
num_samples = len(train_dataset)
if num_samples < 50:
    num_epochs = 50
    batch_size = 4
elif num_samples < 200:
    num_epochs = 30
    batch_size = 8
else:
    num_epochs = 20
    batch_size = 8

print(f"Muestras: {num_samples} | Epochs: {num_epochs} | Batch: {batch_size}")

training_args = TrainingArguments(
    output_dir='/content/mms_kiche_finetuned',
    num_train_epochs=num_epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=2,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    learning_rate=1e-4,
    warmup_steps=50,
    fp16=True,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    push_to_hub=False,
    report_to=['tensorboard'],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=processor,
    compute_metrics=compute_metrics,
)

In [ ]:
# ¡ENTRENAR!
print("Iniciando entrenamiento... ☕")
trainer.train()
print("\n✅ ¡Entrenamiento completado!")

## 8. Evaluar el modelo

In [ ]:
# Ver métricas finales
results = trainer.evaluate()
print(f"\nWER (Word Error Rate): {results['eval_wer']:.2%}")
print("(Más bajo = mejor. <30% es bueno para idiomas de bajos recursos)")

# Probar con algunas muestras
print("\n--- Pruebas ---")
for i in range(min(5, len(eval_dataset))):
    input_values = eval_dataset[i]['input_values']
    input_tensor = torch.tensor(input_values).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_tensor).logits

    pred_ids = torch.argmax(logits, dim=-1)[0]
    prediction = processor.decode(pred_ids)

    # Obtener etiqueta real
    label_ids = eval_dataset[i]['labels']
    label_ids_clean = [l for l in label_ids if l != -100]
    reference = processor.decode(label_ids_clean)

    print(f"  Real:      {reference}")
    print(f"  Predicho:  {prediction}")
    print()

## 9. Guardar modelo entrenado

In [ ]:
import shutil

OUTPUT_DIR = '/content/mms_kiche_trained'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Guardar modelo LoRA (solo los pesos adaptados, mucho más pequeño)
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

# Ver tamaño
total_size = sum(os.path.getsize(os.path.join(dp, f))
                 for dp, dn, files in os.walk(OUTPUT_DIR)
                 for f in files)
print(f"Tamaño del modelo LoRA: {total_size / 1024 / 1024:.1f} MB")
print(f"(vs ~3.8GB del modelo completo — LoRA es mucho más ligero)")

# Comprimir para descargar
shutil.make_archive('/content/mms_kiche_trained', 'zip', OUTPUT_DIR)
print(f"\n✅ Modelo guardado en: /content/mms_kiche_trained.zip")

In [ ]:
# === OPCIÓN A: Descargar directamente ===
from google.colab import files
files.download('/content/mms_kiche_trained.zip')

# === OPCIÓN B: Guardar en Google Drive ===
# DRIVE_OUTPUT = '/content/drive/MyDrive/traductor_kiche/mms_kiche_trained.zip'
# shutil.copy('/content/mms_kiche_trained.zip', DRIVE_OUTPUT)
# print(f"Guardado en Drive: {DRIVE_OUTPUT}")

## 10. Descargar modelo TTS K'iche' (bonus)

Meta ya tiene un modelo TTS para K'iche'. Lo descargamos para que tu app pueda hablar K'iche' localmente.

In [ ]:
from transformers import VitsModel, AutoTokenizer

TTS_MODEL_ID = 'facebook/mms-tts-quc'
TTS_OUTPUT_DIR = '/content/mms_tts_kiche'

print("Descargando modelo TTS K'iche'...")
tts_model = VitsModel.from_pretrained(TTS_MODEL_ID)
tts_tokenizer = AutoTokenizer.from_pretrained(TTS_MODEL_ID)

tts_model.save_pretrained(TTS_OUTPUT_DIR)
tts_tokenizer.save_pretrained(TTS_OUTPUT_DIR)

# Comprimir
shutil.make_archive('/content/mms_tts_kiche', 'zip', TTS_OUTPUT_DIR)
print("\n✅ Modelo TTS K'iche' guardado.")

# Probar que funcione
test_text = "sib'alaj maltyox"
inputs = tts_tokenizer(test_text, return_tensors='pt')
with torch.no_grad():
    output = tts_model(**inputs)
waveform = output.waveform[0].cpu().numpy()
print(f"Audio generado: {len(waveform)} samples ({len(waveform)/tts_model.config.sampling_rate:.1f}s)")

# Reproducir
from IPython.display import Audio
Audio(waveform, rate=tts_model.config.sampling_rate)

In [ ]:
# Descargar TTS
files.download('/content/mms_tts_kiche.zip')

print("\n" + "="*50)
print("RESUMEN - Archivos para descargar:")
print("="*50)
print("1. mms_kiche_trained.zip  → Modelo ASR afinado (reconoce voz K'iche')")
print("2. mms_tts_kiche.zip      → Modelo TTS (habla K'iche')")
print("\nCopialos a tu carpeta traductor_kiche/models/ en tu Mac")
print("y ejecuta: python load_trained_model.py")